In [10]:
### ============ 1. IMPORTS =====================
import pandas as pd # for data manipulation
import pickle # for saving the processed data to disk
import sys # for manipulating the Python path
import os # for file path manipulations
from scipy.io import arff # for loading ARFF files from OpenML

# Make the src/ package importable from within the notebooks/ subfolder.
sys.path.append("..")

# Pull in the same SEED and TEST_SIZE constants from config.py so that all notebooks use the same values.
from src.config import SEED, TEST_SIZE
from src.preprocess import split_data, scale_data

In [ ]:
# DATASET_NAME = "dataset1"
# FILE_PATH = f"../data/raw/{DATASET_NAME}.arff"
# TARGET_COL = "CLASS_LABEL" 

In [11]:
# ========================== 3. LOAD DATA =====================
# Load the data files
# DATASET 1
data, meta = arff.loadarff('../data/raw/Phishing_Legitimate_full.arff')
# Convert to a Pandas data frame
df = pd.DataFrame(data)
# inspect the data
df.head()

# DATASET 2
df = pd.read_csv('../data/raw/dataset_full.csv')
# inspect the data
print(df.shape)
df.head()

,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,NumPercent,...,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
0,3.0,1.0,5.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,1.0,-1.0,1.0,b'1'
1,3.0,1.0,3.0,144.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,1.0,b'1'
2,3.0,1.0,2.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,-1.0,1.0,-1.0,0.0,b'1'
3,3.0,1.0,6.0,79.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,-1.0,b'1'
4,3.0,0.0,4.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,-1.0,0.0,-1.0,-1.0,b'1'


In [12]:
# ===== 3. DATA INSPECTION =====
# check the shape to confirm we have the expected number of rows and columns
print(df.shape)
# show summary statistics and data types to check for any obvious issues
print(df.info()) 
# check for missing values and duplicates
print(df.isnull().sum())
#  check if there are duplicates so they can be removed before training the GA
print(df.duplicated().sum())

(10000, 49)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 49 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   NumDots                             10000 non-null  float64
 1   SubdomainLevel                      10000 non-null  float64
 2   PathLevel                           10000 non-null  float64
 3   UrlLength                           10000 non-null  float64
 4   NumDash                             10000 non-null  float64
 5   NumDashInHostname                   10000 non-null  float64
 6   AtSymbol                            10000 non-null  float64
 7   TildeSymbol                         10000 non-null  float64
 8   NumUnderscore                       10000 non-null  float64
 9   NumPercent                          10000 non-null  float64
 10  NumQueryComponents                  10000 non-null  float64
 11  NumAmpersand                        10000

In [6]:
# ===== 4. PREPROCESSING =====
### Identify target and features
target_col = "CLASS_LABEL"

# Drop the label column to get the feature matrix X.
X = df.drop(columns=[target_col])

# convert bytes to int
y = df["CLASS_LABEL"].astype(str).astype(int)

print("X shape:", X.shape[1])   # number of features
print("y shape:", y.shape)      # number of samples

# check the distribution of the target variable
print(y.value_counts())
print(y.unique())

X shape: 48
y shape: (10000,)
CLASS_LABEL
1    5000
0    5000
Name: count, dtype: int64
[1 0]


In [13]:
# ===== 6. TRAIN / TEST SPLIT =====
# Use the same SEED and TEST_SIZE constants from config.py to ensure all notebooks use the same split.
X_train, X_test, y_train, y_test = split_data(X, y, test_size=TEST_SIZE, seed=SEED)
print(X_train.shape, X_test.shape)  # confirm 70/30 ratio looks right

(7000, 48) (3000, 48)


In [14]:
# ===== 7. SCALE FEATURES =====
# scale the data
X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

# Verify the scaling worked by checking the min and max values of the scaled features.
print(X_train_scaled.describe().loc[["min", "max"]])

     NumDots  SubdomainLevel  PathLevel  UrlLength  NumDash  \
min      0.0             0.0        0.0        0.0      0.0   
max      1.0             1.0        1.0        1.0      1.0   

     NumDashInHostname  AtSymbol  TildeSymbol  NumUnderscore  NumPercent  ...  \
min                0.0       0.0          0.0            0.0         0.0  ...   
max                1.0       1.0          1.0            1.0         1.0  ...   

     SubmitInfoToEmail  IframeOrFrame  MissingTitle  ImagesOnlyInForm  \
min                0.0            0.0           0.0               0.0   
max                1.0            1.0           1.0               1.0   

     SubdomainLevelRT  UrlLengthRT  PctExtResourceUrlsRT  \
min               0.0          0.0                   0.0   
max               1.0          1.0                   1.0   

     AbnormalExtFormActionR  ExtMetaScriptLinkRT  \
min                     0.0                  0.0   
max                     1.0                  1.0   

     Pct

In [17]:
# ===== 8. SAVE PROCESSED DATA =====
# Persist the split and scaled arrays to disk with pickle so that
# subsequent notebooks (02, 03, 04) can reload them without re-running
# the entire preprocessing pipeline from scratch.
with open("../data/processed/dataset1_split_scaled.pkl", "wb") as f:
    pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test), f)